# Evaluación baseline: falla de prompting directo (los tres candidatos)

Este notebook mide, con evidencia propia, qué tan seguido falla cada uno de los
tres modelos candidatos cuando se les pide directamente (sin fine-tuning ni
técnicas adicionales) generar el SQL necesario para responder preguntas de
negocio sobre `business.db`.

Corresponde a la evidencia propia que exige el punto "Failure diagnosis" de la
rúbrica del Entregable 1, además de las cifras ya citadas de la literatura.

Requiere GPU (Runtime > Change runtime type > T4 GPU en Colab). Corre los tres
modelos en secuencia dentro de la misma sesión, liberando memoria entre uno y
otro.

In [1]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.9 MB/s eta 0:00:00


## 0. Acceso a Llama-3.1-8B-Instruct

Llama-3.1-8B-Instruct es un modelo con acceso restringido ("gated") en Hugging
Face.

Qwen2.5-Coder-7B-Instruct y Qwen2.5-Coder-3B-Instruct no requieren esto.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Cargar la base de datos y las preguntas

Sube `business.db` y `questions.json` (carpeta `data/` del repositorio) a esta sesión de Colab, o móntalos desde Google Drive.

In [3]:
import json
import sqlite3
import re
import gc
from pathlib import Path

DB_PATH = "business.db"
Q_PATH = "questions.json"

data = json.loads(Path(Q_PATH).read_text(encoding="utf-8"))
SCHEMA = data["schema"]
QUESTIONS = data["questions"]
print(f"{len(QUESTIONS)} preguntas cargadas.")
print(SCHEMA)

15 preguntas cargadas.
CREATE TABLE products (product_id INTEGER PRIMARY KEY, name TEXT, category TEXT, price INTEGER);
CREATE TABLE sales (sale_id INTEGER PRIMARY KEY, product_id INTEGER, sale_date TEXT, quantity INTEGER, amount INTEGER);
CREATE TABLE inventory (product_id INTEGER PRIMARY KEY, stock INTEGER, reorder_point INTEGER);


## 2. Los tres modelos candidatos a comparar

In [4]:
MODEL_CANDIDATES = [
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Qwen/Qwen2.5-Coder-3B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
]

## 3. Prompt de prompting directo (baseline, sin técnicas adicionales) y utilidades de ejecución SQL

In [16]:
PROMPT_TEMPLATE = """Eres un asistente que traduce preguntas de negocio a SQL.

Esquema de la base de datos:
{schema}

Pregunta: {question}

Responde unicamente con la o las consultas SQL necesarias para responder la
pregunta, separadas por punto y coma. No expliques nada, no uses markdown."""


def extract_sql_statements(text):
    text = re.sub(r"```sql|```", "", text, flags=re.IGNORECASE)
    parts = [p.strip() for p in text.split(";")]
    return [p for p in parts if p and p.upper().startswith("SELECT")]


def run_sql(conn, sql):
    try:
        cur = conn.cursor()
        cur.execute(sql)
        return cur.fetchall()
    except Exception as e:
        return f"ERROR: {e}"


def matches_gold(generated_results, gold_result):
    def normalize(rows):
        return set(str(tuple(row)) for row in rows)
    gold_set = [normalize(r) for r in gold_result]
    gen_set = [normalize(r) for r in generated_results if isinstance(r, list)]
    return all(any(g == gset for gset in gen_set) for g in gold_set)

In [17]:
import gc, torch
try:
    del model, tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()


## 4. Loop principal: carga cada modelo, evalua, libera memoria, sigue con el siguiente

In [18]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

all_results = {}

for model_name in MODEL_CANDIDATES:
    print(f"\n=== Cargando {model_name} ===")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb_config, device_map="auto"
    )

    def ask_model(question, tok=tokenizer, mdl=model):
        prompt = PROMPT_TEMPLATE.format(schema=SCHEMA, question=question)
        messages = [{"role": "user", "content": prompt}]
        inputs = tok.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(mdl.device)
        output = mdl.generate(
            **inputs, max_new_tokens=300, do_sample=False, temperature=None, top_p=None
        )
        text = tok.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return text.strip()

    conn = sqlite3.connect(DB_PATH)
    results = []
    for q in QUESTIONS:
        raw_output = ask_model(q["question"])
        statements = extract_sql_statements(raw_output)
        executed = [run_sql(conn, s) for s in statements]
        correct = matches_gold(executed, q["gold_result"])
        results.append({
            "id": q["id"],
            "type": q["type"],
            "question": q["question"],
            "model_output": raw_output,
            "n_statements_generated": len(statements),
            "correct": correct,
        })
        print(f"  {q['id']} [{q['type']}] correcto={correct}")
    conn.close()

    all_results[model_name] = results

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

print("\nListo, los tres modelos fueron evaluados.")


=== Cargando Qwen/Qwen2.5-Coder-7B-Instruct ===


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

  q01 [puntual] correcto=True
  q02 [puntual] correcto=False
  q03 [puntual] correcto=False
  q04 [puntual] correcto=True
  q05 [puntual] correcto=True
  q06 [puntual] correcto=False
  q07 [puntual] correcto=False
  q08 [puntual] correcto=True
  q09 [combinada] correcto=False
  q10 [combinada] correcto=False
  q11 [combinada] correcto=False
  q12 [combinada] correcto=False
  q13 [puntual] correcto=True
  q14 [combinada] correcto=False
  q15 [puntual] correcto=False

=== Cargando Qwen/Qwen2.5-Coder-3B-Instruct ===


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  q01 [puntual] correcto=True
  q02 [puntual] correcto=False
  q03 [puntual] correcto=False
  q04 [puntual] correcto=True
  q05 [puntual] correcto=True
  q06 [puntual] correcto=True
  q07 [puntual] correcto=True
  q08 [puntual] correcto=True
  q09 [combinada] correcto=False
  q10 [combinada] correcto=False
  q11 [combinada] correcto=False
  q12 [combinada] correcto=False
  q13 [puntual] correcto=True
  q14 [combinada] correcto=False
  q15 [puntual] correcto=False

Listo, los tres modelos fueron evaluados.


In [10]:
q = QUESTIONS[0]
prompt = PROMPT_TEMPLATE.format(schema=SCHEMA, question=q["question"])
messages = [{"role": "user", "content": prompt}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)
output = model.generate(**inputs, max_new_tokens=300, do_sample=False)
text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(repr(text))


"SELECT SUM(quantity) FROM sales WHERE product_id = (SELECT product_id FROM products WHERE name = 'Notebook 14 pulgadas') AND sale_date LIKE '2026-06-%';"


In [14]:
q = QUESTIONS[0]
statements = extract_sql_statements(text)
print("Statements extraidos:", statements)

executed = [run_sql(conn, s) for s in statements]
print("Resultado ejecutado:", executed)

print("Resultado esperado (gold):", q["gold_result"])

print("Matches:", matches_gold(executed, q["gold_result"]))


Statements extraidos: ["SELECT SUM(quantity) FROM sales WHERE product_id = (SELECT product_id FROM products WHERE name = 'Notebook 14 pulgadas') AND sale_date LIKE '2026-06-%'"]
Resultado ejecutado: [[(34,)]]
Resultado esperado (gold): [[[34]]]
Matches: False


## 5. Tabla comparativa (esta es la evidencia propia para el documento)

In [19]:
import pandas as pd

rows = []
for model_name, results in all_results.items():
    df = pd.DataFrame(results)
    overall = df["correct"].mean()
    puntual = df[df["type"] == "puntual"]["correct"].mean()
    combinada = df[df["type"] == "combinada"]["correct"].mean()
    rows.append({
        "modelo": model_name,
        "exactitud_global": overall,
        "exactitud_puntual": puntual,
        "exactitud_combinada": combinada,
    })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

comparison.to_csv("baseline_comparison.csv", index=False)

with open("baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("\nGuardado baseline_comparison.csv y baseline_results.json")

                          modelo  exactitud_global  exactitud_puntual  exactitud_combinada
  Qwen/Qwen2.5-Coder-7B-Instruct          0.333333                0.5                  0.0
  Qwen/Qwen2.5-Coder-3B-Instruct          0.466667                0.7                  0.0
meta-llama/Llama-3.1-8B-Instruct          0.466667                0.7                  0.0

Guardado baseline_comparison.csv y baseline_results.json
